In [ ]:
"""
Project 01: Credit Risk Prediction
Step 06: Additional Model Training (RandomForest, GradientBoosting, ExtraTrees, AdaBoost)
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle
import os
import json
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
)

In [ ]:
try:
    from catboost import CatBoostClassifier

    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("CatBoost not available, skipping...")
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
)

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if "__file__" in globals() else (os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 06: Additional Model Training")
print("=" * 60)
start_time = time.time()

In [ ]:
# Load Preprocessed Data
print("\nLoading preprocessed data...")
df = pd.read_csv(os.path.join(PROCESSED, "preprocessed.csv"))
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
TARGET = "TARGET"
ID_COL = "SK_ID_CURR"
X = df.drop(columns=[TARGET, ID_COL], errors="ignore")
y = df[TARGET]

In [ ]:
print(f"Target distribution:\n{y.value_counts()}")
print(f"Default rate: {y.mean():.2%}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

In [ ]:
feature_names = X.columns.tolist()

In [ ]:
def evaluate(name, model, X_te, y_te):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    metrics = {
        "Model": name,
        "auroc": round(roc_auc_score(y_te, y_proba), 4),
        "balanced_accuracy": round(balanced_accuracy_score(y_te, y_pred), 4),
        "f1": round(f1_score(y_te, y_pred), 4),
        "precision": round(precision_score(y_te, y_pred), 4),
        "recall": round(recall_score(y_te, y_pred), 4),
    }
    print(f"\n  {name}:")
    for k, v in metrics.items():
        if k != "Model":
            print(f"    {k}: {v}")
    return metrics

In [ ]:
results_path = os.path.join(MODELS, "results.json")
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print(f"Loaded existing results with {len(results)} models")
else:
    results = {}

In [ ]:
scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

In [ ]:
# Model 4: Random Forest
print("\n[4/5] Training Random Forest...")
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
rf_time = round(time.time() - t0, 2)
metrics_rf = evaluate("Random Forest", rf, X_test, y_test)
metrics_rf["train_time_sec"] = rf_time

In [ ]:
with open(os.path.join(MODELS, "random_forest.pkl"), "wb") as f:
    pickle.dump({"model": rf, "feature_names": feature_names}, f)
print(f"  Saved | Train time: {rf_time}s")
results["Random Forest"] = {k: v for k, v in metrics_rf.items() if k != "Model"}

In [ ]:
# Model 5: Gradient Boosting
print("\n[5/5] Training Gradient Boosting...")
t0 = time.time()
gb = GradientBoostingClassifier(
    n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42
)
gb.fit(X_train, y_train)
gb_time = round(time.time() - t0, 2)
metrics_gb = evaluate("Gradient Boosting", gb, X_test, y_test)
metrics_gb["train_time_sec"] = gb_time

In [ ]:
with open(os.path.join(MODELS, "gradient_boosting.pkl"), "wb") as f:
    pickle.dump({"model": gb, "feature_names": feature_names}, f)
print(f"  Saved | Train time: {gb_time}s")
results["Gradient Boosting"] = {k: v for k, v in metrics_gb.items() if k != "Model"}

In [ ]:
# Model 6: Extra Trees
print("\n[6/5] Training Extra Trees...")
t0 = time.time()
et = ExtraTreesClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
et.fit(X_train, y_train)
et_time = round(time.time() - t0, 2)
metrics_et = evaluate("Extra Trees", et, X_test, y_test)
metrics_et["train_time_sec"] = et_time

In [ ]:
with open(os.path.join(MODELS, "extra_trees.pkl"), "wb") as f:
    pickle.dump({"model": et, "feature_names": feature_names}, f)
print(f"  Saved | Train time: {et_time}s")
results["Extra Trees"] = {k: v for k, v in metrics_et.items() if k != "Model"}

In [ ]:
# Model 7: AdaBoost
print("\n[7/5] Training AdaBoost...")
t0 = time.time()
ada = AdaBoostClassifier(n_estimators=100, learning_rate=0.5, random_state=42)
ada.fit(X_train, y_train)
ada_time = round(time.time() - t0, 2)
metrics_ada = evaluate("AdaBoost", ada, X_test, y_test)
metrics_ada["train_time_sec"] = ada_time

In [ ]:
with open(os.path.join(MODELS, "adaboost.pkl"), "wb") as f:
    pickle.dump({"model": ada, "feature_names": feature_names}, f)
print(f"  Saved | Train time: {ada_time}s")
results["AdaBoost"] = {k: v for k, v in metrics_ada.items() if k != "Model"}

In [ ]:
# Model 8: CatBoost (if available)
if CATBOOST_AVAILABLE:
    print("\n[8/5] Training CatBoost...")
    t0 = time.time()
    cat = CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        auto_class_weights="Balanced",
        random_state=42,
        verbose=0,
    )
    cat.fit(X_train, y_train)
    cat_time = round(time.time() - t0, 2)
    metrics_cat = evaluate("CatBoost", cat, X_test, y_test)
    metrics_cat["train_time_sec"] = cat_time

    with open(os.path.join(MODELS, "catboost.pkl"), "wb") as f:
        pickle.dump({"model": cat, "feature_names": feature_names}, f)
    print(f"  Saved | Train time: {cat_time}s")
    results["CatBoost"] = {k: v for k, v in metrics_cat.items() if k != "Model"}

In [ ]:
# Save updated results
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults updated: {results_path}")

In [ ]:
# Chart: All Models Comparison
print("\nGenerating model comparison chart...")
df_all = pd.DataFrame(
    [
        {
            "Model": k,
            "auroc": v.get("auroc", 0),
            "f1": v.get("f1", 0),
            "precision": v.get("precision", 0),
            "recall": v.get("recall", 0),
        }
        for k, v in results.items()
    ]
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

In [ ]:
for idx, (metric, title, color) in enumerate(
    [
        ("auroc", "AUROC (Higher = Better)", "#43E97B"),
        ("f1", "F1 Score (Higher = Better)", "#6C63FF"),
        ("precision", "Precision (Higher = Better)", "#FF6584"),
        ("recall", "Recall (Higher = Better)", "#F9A826"),
    ]
):
    ax = axes[idx]
    df_sorted = df_all.sort_values(metric, ascending=False)
    bars = ax.bar(df_sorted["Model"], df_sorted[metric], color=color, alpha=0.85)
    ax.set_title(title, fontweight="bold")
    ax.set_ylim(0, 1.1)
    ax.tick_params(axis="x", rotation=45)
    for bar, val in zip(bars, df_sorted[metric]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{val:.3f}",
            ha="center",
            fontsize=8,
        )

In [ ]:
plt.suptitle("All Models Comparison — Credit Risk", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(
    os.path.join(CHARTS, "all_models_comparison.png"), dpi=150, bbox_inches="tight"
)
plt.close()
print("  Saved: all_models_comparison.png")

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 06 COMPLETE | Time: {elapsed:.1f}s")
print(f"Trained 4 new models: Random Forest, Gradient Boosting, Extra Trees, AdaBoost")
print(f"Total models: {len(results)}")
print(f"{'=' * 60}")